### APEX WEALTH DATA PIPELINE


In [1]:
!pip install requests

In [2]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

In [3]:

load_dotenv()
api_key = os.getenv('API_KEY')

In [4]:
API_KEY = os.getenv('API_KEY')

In [5]:
api_key

'0fb28af206c64e9da726186651ad43c2'

In [6]:
symbol = "AAPL"
url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=20"


In [7]:
#Get response from the API
response = requests.get(url)
response.status_code

200

In [8]:
url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=20"
params = {
    "symbol": "AAPL",
    "interval": "1day",
    "apikey": API_KEY
}


In [9]:


resp = requests.get(url, params=params)
print("Status code:", resp.status_code)  # Should be 200

data = resp.json()
print("Keys in JSON:", data.keys())      # Should include 'values'


Status code: 200
Keys in JSON: dict_keys(['meta', 'values', 'status'])


In [10]:
#get data in json format
data = response.json()
data


{'meta': {'symbol': 'AAPL',
  'interval': '1min',
  'currency': 'USD',
  'exchange_timezone': 'America/New_York',
  'exchange': 'NASDAQ',
  'mic_code': 'XNGS',
  'type': 'Common Stock'},
 'values': [{'datetime': '2026-01-30 15:59:00',
   'open': '259.88000',
   'high': '260.06',
   'low': '258',
   'close': '259.48001',
   'volume': '1973068'},
  {'datetime': '2026-01-30 15:58:00',
   'open': '259.60001',
   'high': '259.93',
   'low': '259.51501',
   'close': '259.87000',
   'volume': '683270'},
  {'datetime': '2026-01-30 15:57:00',
   'open': '259.87000',
   'high': '259.88',
   'low': '259.36499',
   'close': '259.57001',
   'volume': '574705'},
  {'datetime': '2026-01-30 15:56:00',
   'open': '259.82999',
   'high': '260.040009',
   'low': '259.66000',
   'close': '259.85999',
   'volume': '375311'},
  {'datetime': '2026-01-30 15:55:00',
   'open': '259.79999',
   'high': '260.019989',
   'low': '258.89999',
   'close': '259.82999',
   'volume': '603386'},
  {'datetime': '2026-01-3

In [11]:
df = pd.DataFrame(data["values"])
df.head()

,datetime,open,high,low,close,volume
0,2026-01-30 15:59:00,259.88000,260.06,258,259.48001,1973068
1,2026-01-30 15:58:00,259.60001,259.93,259.51501,259.87000,683270
2,2026-01-30 15:57:00,259.87000,259.88,259.36499,259.57001,574705
3,2026-01-30 15:56:00,259.82999,260.040009,259.66000,259.85999,375311
4,2026-01-30 15:55:00,259.79999,260.019989,258.89999,259.82999,603386


In [12]:
#transform data to dataframe
def transform_data(data):
    #extract values
    time_series = data['values']

    #convert to dataframe
    df = pd.DataFrame(time_series)

    #convert to proper datatypes
    df['datetime'] = pd.to_datetime(df['datetime'])

    df = df.astype({
        'open': 'float',
        'high': 'float',
        'low': 'float',
        'close': 'float',
        'volume': 'int' })
    return df


In [13]:
df = transform_data(data)

In [14]:
df

,datetime,open,high,low,close,volume
0,2026-01-30 15:59:00,259.880000,260.060000,258.000000,259.480010,1973068
1,2026-01-30 15:58:00,259.600010,259.930000,259.515010,259.870000,683270
2,2026-01-30 15:57:00,259.870000,259.880000,259.364990,259.570010,574705
3,2026-01-30 15:56:00,259.829990,260.040009,259.660000,259.859990,375311
4,2026-01-30 15:55:00,259.799990,260.019989,258.899990,259.829990,603386
5,2026-01-30 15:54:00,259.950010,260.089996,259.060089,259.790010,463393
6,2026-01-30 15:53:00,260.769990,260.790010,259.850010,259.950010,300682
7,2026-01-30 15:52:00,261.019989,261.100010,260.590000,260.769010,407026
8,2026-01-30 15:51:00,261.419890,261.490000,260.910000,261.024994,442079
9,2026-01-30 15:50:00,261.739990,261.890010,261.380000,261.410000,832226


### Extract data for multiple symbols

In [15]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
all_data = []
def fetch_data(symbol, api_key):
    url =f"https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=5"
    response = requests.get(url)
    response.raise_for_status() #raise an error if we get a bad response
    data = response.json()

    if data.get('status') != 'ok':
        raise ValueError(f'Error fetching data for {symbol}: {data.get("message", "Unknown error")}')
    return data

In [16]:
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']

for symbol in symbols:
    data = fetch_data(symbol, api_key)
    df = transform_data(data)
    df['symbol'] = symbol
    all_data.append(df)

all_data


[             datetime       open        high        low      close   volume  \
 0 2026-01-30 15:59:00  259.88000  260.060000  258.00000  259.48001  1973068   
 1 2026-01-30 15:58:00  259.60001  259.930000  259.51501  259.87000   683270   
 2 2026-01-30 15:57:00  259.87000  259.880000  259.36499  259.57001   574705   
 3 2026-01-30 15:56:00  259.82999  260.040009  259.66000  259.85999   375311   
 4 2026-01-30 15:55:00  259.79999  260.019989  258.89999  259.82999   603386   
 
   symbol  
 0   AAPL  
 1   AAPL  
 2   AAPL  
 3   AAPL  
 4   AAPL  ,
              datetime        open       high        low      close  volume  \
 0 2026-01-30 15:59:00  429.900090  430.42001  429.49011  430.39001  846171   
 1 2026-01-30 15:58:00  429.685000  429.94000  429.35000  429.90500  346721   
 2 2026-01-30 15:57:00  429.670010  429.98001  429.53000  429.69000  293137   
 3 2026-01-30 15:56:00  429.340000  429.70639  429.23999  429.67999  283368   
 4 2026-01-30 15:55:00  428.019989  429.37000  427

In [17]:
#concat - joining or stacking all the dataframes together
all_df = pd.concat(all_data, ignore_index=True)

In [18]:
all_df

,datetime,open,high,low,close,volume,symbol
0,2026-01-30 15:59:00,259.880000,260.060000,258.00000,259.480010,1973068,AAPL
1,2026-01-30 15:58:00,259.600010,259.930000,259.51501,259.870000,683270,AAPL
2,2026-01-30 15:57:00,259.870000,259.880000,259.36499,259.570010,574705,AAPL
3,2026-01-30 15:56:00,259.829990,260.040009,259.66000,259.859990,375311,AAPL
4,2026-01-30 15:55:00,259.799990,260.019989,258.89999,259.829990,603386,AAPL
5,2026-01-30 15:59:00,429.900090,430.420010,429.49011,430.390010,846171,MSFT
6,2026-01-30 15:58:00,429.685000,429.940000,429.35000,429.905000,346721,MSFT
7,2026-01-30 15:57:00,429.670010,429.980010,429.53000,429.690000,293137,MSFT
8,2026-01-30 15:56:00,429.340000,429.706390,429.23999,429.679990,283368,MSFT
9,2026-01-30 15:55:00,428.019989,429.370000,427.98500,429.337710,329935,MSFT


#### Loading

In [33]:
load_dotenv()
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')


In [1]:
import os
from dotenv import load_dotenv

# Load the .env file
load_dotenv(dotenv_path="C:/full/path/to/your/project/.env")  # change this to your actual path

# Get DB_NAME
DB_NAME = os.getenv("DB_NAME")
print("DB_NAME:", DB_NAME)


DB_NAME: None


In [2]:
print(DB_NAME)

None


In [27]:
#create a database connnection url 

from sqlalchemy import create_engine 
import psycopg2


db_url = f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(db_url)

#load dataframe to postgres database
all_df.to_sql('stockPrices_data', engine, if_exists='append', index=False)

print("Data loaded to database successfully")             

ValueError: invalid literal for int() with base 10: 'None'

In [22]:
all_df.to_csv('stock_data.csv', index=False)

In [ ]:
sa